# 📈 Stocks — Data Collector
**Data source:** Yahoo Finance &nbsp;·&nbsp; `yfinance` Python library

This notebook lets you download historical data for individual **stocks**.

### What you can download
| Category | What it contains |
|---|---|
| 📊 Prices | Adjusted close price history |
| 📈 Returns | Period-over-period percentage returns |
| 📈 Cumulative Returns | Total return since start of the period |
| 🏢 Company Info | Sector, industry, country, market cap |
| 💰 Valuation Ratios | P/E, P/B, EV/EBITDA, ROE, margins, beta… |
| 💵 Dividends | Historical dividend payment dates and amounts |
| 🌱 ESG Scores | Environmental, Social and Governance ratings |
| 📄 Income Statement | Revenue, operating profit, net income… |
| 📄 Balance Sheet | Assets, liabilities, equity… |
| 📄 Cash Flow Statement | Operating, investing, financing cash flows… |

### How to use this notebook
1. **Run the Setup cell** (▶ or `Shift + Enter`) — do this first, every time
2. Enter your tickers and set the date range
3. Select the data you want
4. Click **Download Data**

The output Excel file is saved in the `data/` folder at the root of the project.  
If you also run the Indexes or Macro notebooks, all data ends up in the **same Excel file**, each level on its own sheet.


In [ ]:
# ── Setup — Run this cell first ─────────────────────────────────────────────
# This cell loads all the tools needed for the notebook.
# You do not need to edit anything here.

import sys
import datetime
from pathlib import Path

# Make the src/ folder importable from this notebook
sys.path.insert(0, str(Path("..").resolve()))

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

from src.collect import (
    download_prices,
    get_company_info,
    get_valuation_ratios,
    get_dividends,
    get_esg_scores,
    get_financial_statement,
)
from src.transform import resample_prices, calculate_returns, calculate_cumulative_returns
from src.export import save_to_excel

print("✅ Setup complete — continue to the next cell.")


---
## Step 1 — Enter Stock Tickers

A **ticker** is the short code used to identify a stock on an exchange.

💡 To find a ticker: go to [finance.yahoo.com](https://finance.yahoo.com), search for the company name, and copy the symbol shown at the top of the page.

**Examples**
- US stocks: `AAPL` (Apple), `MSFT` (Microsoft), `TSLA` (Tesla), `JPM` (JPMorgan)
- European stocks: `MC.PA` (LVMH), `SIE.DE` (Siemens), `NESN.SW` (Nestlé)
- Enter multiple tickers separated by commas


In [ ]:
ticker_input = widgets.Textarea(
    value="AAPL, MSFT, GOOGL",
    placeholder="e.g. AAPL, MSFT, TSLA, MC.PA",
    layout=widgets.Layout(width="500px", height="65px"),
)

display(
    widgets.HTML("<b>Tickers:</b>"),
    ticker_input,
    widgets.HTML(
        "<small style='color:#666'>Separate multiple tickers with commas. "
        "Use the exact Yahoo Finance format (e.g. MC.PA for Paris-listed stocks).</small>"
    ),
)


---
## Step 2 — Date Range & Frequency

Choose the time period and how often you want observations.

| Frequency | Best used for |
|---|---|
| Daily | Short-term analysis, volatility studies |
| Monthly | Medium-term trends, most academic studies |
| Quarterly | Aligns with financial statement reporting periods |
| Yearly | Long-run historical comparisons |

⚠️ **Note:** For financial statements (Income Statement, Balance Sheet, Cash Flow),
frequency and date range apply only to price data.
Financial statements are always shown for the last 4 annual (or quarterly) periods available.


In [ ]:
start_date = widgets.DatePicker(
    description="Start date:",
    value=datetime.date(2020, 1, 1),
    style={"description_width": "90px"},
)
end_date = widgets.DatePicker(
    description="End date:",
    value=datetime.date.today(),
    style={"description_width": "90px"},
)
frequency = widgets.Dropdown(
    options=["Daily", "Monthly", "Quarterly", "Yearly"],
    value="Monthly",
    description="Frequency:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="220px"),
)

display(
    widgets.HTML("<b>Date range:</b>"),
    widgets.HBox([start_date, end_date]),
    widgets.HTML("<br><b>Frequency:</b>"),
    frequency,
)


---
## Step 3 — Choose Data to Download

Select everything you need. You can always re-run the notebook to download additional data.


In [ ]:
cb_prices     = widgets.Checkbox(value=True,  description="📊  Prices (adjusted close)")
cb_returns    = widgets.Checkbox(value=True,  description="📈  Returns (period-over-period)")
cb_cumreturns = widgets.Checkbox(value=False, description="📈  Cumulative Returns")

cb_info       = widgets.Checkbox(value=False, description="🏢  Company Info (sector, market cap, description…)")
cb_ratios     = widgets.Checkbox(value=False, description="💰  Valuation Ratios (P/E, P/B, EV/EBITDA, ROE…)")
cb_dividends  = widgets.Checkbox(value=False, description="💵  Dividends (payment history)")
cb_esg        = widgets.Checkbox(value=False, description="🌱  ESG Scores (Sustainalytics via Yahoo Finance)")

cb_income     = widgets.Checkbox(value=False, description="📄  Income Statement (annual)")
cb_balance    = widgets.Checkbox(value=False, description="📄  Balance Sheet (annual)")
cb_cashflow   = widgets.Checkbox(value=False, description="📄  Cash Flow Statement (annual)")

display(
    widgets.HTML("<b>Price series:</b>"),
    cb_prices, cb_returns, cb_cumreturns,
    widgets.HTML("<br><b>Company data (point-in-time):</b>"),
    cb_info, cb_ratios, cb_dividends, cb_esg,
    widgets.HTML("<br><b>Financial statements (last 4 annual periods):</b>"),
    cb_income, cb_balance, cb_cashflow,
)


---
## Step 4 — Download & Save

The file will be saved in the `data/` folder.  
If you use the same file name across notebooks, all data will be combined into one Excel file with separate sheets.


In [ ]:
output_file = widgets.Text(
    value="market_data.xlsx",
    description="File name:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="320px"),
)

download_btn = widgets.Button(
    description="⬇  Download Data",
    button_style="success",
    layout=widgets.Layout(width="200px", height="40px"),
)

out = widgets.Output()

display(output_file, download_btn, out)


def on_download(b):
    with out:
        clear_output(wait=True)

        # ── Parse inputs ─────────────────────────────────────────────────────
        raw_tickers = ticker_input.value.strip()
        tickers = [t.strip().upper() for t in raw_tickers.split(",") if t.strip()]

        if not tickers:
            print("❌  Please enter at least one ticker in Step 1.")
            return

        start = str(start_date.value)
        end   = str(end_date.value)
        freq  = frequency.value

        fname = output_file.value.strip() or "market_data.xlsx"
        if not fname.endswith(".xlsx"):
            fname += ".xlsx"

        # ── Check at least one data type selected ────────────────────────────
        any_price = cb_prices.value or cb_returns.value or cb_cumreturns.value
        any_other = any([
            cb_info.value, cb_ratios.value, cb_dividends.value,
            cb_esg.value, cb_income.value, cb_balance.value, cb_cashflow.value,
        ])

        if not any_price and not any_other:
            print("❌  Please select at least one data type in Step 3.")
            return

        output_path = Path("..") / "data" / fname
        sheets = {}

        print(f"Tickers  : {', '.join(tickers)}")
        print(f"Period   : {start}  →  {end}")
        print(f"Frequency: {freq}")
        print()

        # ── Price series ─────────────────────────────────────────────────────
        if any_price:
            print("📊  Downloading prices…")
            try:
                prices = download_prices(tickers, start, end, freq)
                prices = resample_prices(prices, freq)
                print(f"    {len(prices)} rows × {len(prices.columns)} ticker(s)")

                if cb_prices.value:
                    sheets["Prices"] = prices

                if cb_returns.value or cb_cumreturns.value:
                    returns = calculate_returns(prices)
                    if cb_returns.value:
                        sheets["Returns"] = returns
                    if cb_cumreturns.value:
                        sheets["Cumulative Returns"] = calculate_cumulative_returns(returns)

                print("    ✅  Done")
            except Exception as e:
                print(f"    ❌  Error: {e}")

        # ── Company info ──────────────────────────────────────────────────────
        if cb_info.value:
            print("\n🏢  Downloading company info…")
            try:
                df = get_company_info(tickers)
                sheets["Company Info"] = df
                print("    ✅  Done")
            except Exception as e:
                print(f"    ❌  Error: {e}")

        # ── Valuation ratios ──────────────────────────────────────────────────
        if cb_ratios.value:
            print("\n💰  Downloading valuation ratios…")
            try:
                df = get_valuation_ratios(tickers)
                sheets["Valuation Ratios"] = df
                print("    ✅  Done")
            except Exception as e:
                print(f"    ❌  Error: {e}")

        # ── Dividends ─────────────────────────────────────────────────────────
        if cb_dividends.value:
            print("\n💵  Downloading dividends…")
            try:
                df = get_dividends(tickers, start, end)
                if df.empty:
                    print("    ⚠️   No dividend data found for the selected tickers/period.")
                else:
                    sheets["Dividends"] = df
                    print("    ✅  Done")
            except Exception as e:
                print(f"    ❌  Error: {e}")

        # ── ESG ───────────────────────────────────────────────────────────────
        if cb_esg.value:
            print("\n🌱  Downloading ESG scores…")
            try:
                df = get_esg_scores(tickers)
                sheets["ESG Scores"] = df
                print("    ✅  Done")
            except Exception as e:
                print(f"    ❌  Error: {e}")

        # ── Income statement ──────────────────────────────────────────────────
        if cb_income.value:
            print("\n📄  Downloading income statements…")
            try:
                df = get_financial_statement(tickers, "income")
                if df.empty:
                    print("    ⚠️   No income statement data found.")
                else:
                    sheets["Income Statement"] = df
                    print("    ✅  Done")
            except Exception as e:
                print(f"    ❌  Error: {e}")

        # ── Balance sheet ─────────────────────────────────────────────────────
        if cb_balance.value:
            print("\n📄  Downloading balance sheets…")
            try:
                df = get_financial_statement(tickers, "balance")
                if df.empty:
                    print("    ⚠️   No balance sheet data found.")
                else:
                    sheets["Balance Sheet"] = df
                    print("    ✅  Done")
            except Exception as e:
                print(f"    ❌  Error: {e}")

        # ── Cash flow ─────────────────────────────────────────────────────────
        if cb_cashflow.value:
            print("\n📄  Downloading cash flow statements…")
            try:
                df = get_financial_statement(tickers, "cashflow")
                if df.empty:
                    print("    ⚠️   No cash flow data found.")
                else:
                    sheets["Cash Flow"] = df
                    print("    ✅  Done")
            except Exception as e:
                print(f"    ❌  Error: {e}")

        # ── Save to Excel ─────────────────────────────────────────────────────
        if not sheets:
            print("\n❌  No data was collected. Nothing saved.")
            return

        print(f"\n💾  Saving to Excel…")
        try:
            save_to_excel(sheets, output_path)
            print(f"    ✅  Saved: {output_path.resolve()}")
            print(f"    Sheets written: {', '.join(sheets.keys())}")
        except Exception as e:
            print(f"    ❌  Could not save file: {e}")
            return

        # ── Citation ──────────────────────────────────────────────────────────
        from datetime import datetime as dt
        today = dt.today().strftime("%d %B %Y")
        print()
        print("─" * 60)
        print("📋  DATA SOURCE — copy this into your assignment")
        print("─" * 60)
        print(f"Source     : Yahoo Finance (finance.yahoo.com)")
        print(f"Tickers    : {', '.join(tickers)}")
        print(f"Period     : {start} to {end}  |  Frequency: {freq}")
        print(f"Downloaded : {today}")
        print(f"Tool       : yfinance Python library (pypi.org/project/yfinance)")
        print("─" * 60)


download_btn.on_click(on_download)
